# V6 Grid Search — Composite Gate + Selective Mixing + Diagonal Band

**Yenilikler (V5'e gore):**
- Composite gate: cR(0.45) + pLDDT(0.30) + Rg(0.15) + pTM(0.10), threshold=0.55
- pTM hard cutoff: 0.15 (eski 0.30 cok sikiydi)
- Diagonal band masking: |i-j| <= band pozisyonlari trunk'tan korunur

**Fazlar:**
- A: Uniform baseline vs Selective baseline (2 run)
- B: change_cutoff sweep (5 run)
- C: alpha_base x alpha_max (best cutoff ile, 6 run)
- D: mapping & weight sweep (9 run)
- E: diagonal_band sweep (4 run)

**Toplam:** ~26 run

## 1. Environment Setup

In [ ]:
import os, sys, shutil

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

if os.path.exists('/content/ANM-openfold3'):
    shutil.rmtree('/content/ANM-openfold3')
!git clone https://github.com/beratkaskaloglu/ANM-openfold3.git /content/ANM-openfold3

if not os.path.exists('/content/ANM-openfold3/openfold3-repo'):
    !git clone https://github.com/aqlaboratory/openfold-3.git /content/ANM-openfold3/openfold3-repo
    !pip install -e /content/ANM-openfold3/openfold3-repo -q
else:
    try:
        import openfold3
    except ImportError:
        !pip install -e /content/ANM-openfold3/openfold3-repo -q

!pip install -q biopython matplotlib numpy torch pandas

_repo_root = '/content/ANM-openfold3'
if _repo_root not in sys.path:
    sys.path.insert(0, _repo_root)
if '/content/ANM-openfold3/openfold3-repo' not in sys.path:
    sys.path.insert(0, '/content/ANM-openfold3/openfold3-repo')

os.makedirs(os.path.expanduser('~/.openfold3'), exist_ok=True)

from pathlib import Path
from openfold3.entry_points.parameters import (
    download_model_parameters,
    get_default_checkpoint_dir,
    DEFAULT_CHECKPOINT_NAME,
)
_param_dir = get_default_checkpoint_dir()
_param_dir.mkdir(parents=True, exist_ok=True)
download_model_parameters(_param_dir, DEFAULT_CHECKPOINT_NAME, skip_confirmation=True)

print('Environment setup complete.')

## 2. Config + Checkpoint Utils

In [ ]:
import json
from datetime import datetime

# ════════════════════════════════════════════════════
#  DRIVE CHECKPOINT CONFIG
# ════════════════════════════════════════════════════
DRIVE_SAVE_DIR = '/content/drive/MyDrive/ANM-openfold3/search_v6'
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)


def checkpoint_path(phase_name: str) -> str:
    return os.path.join(DRIVE_SAVE_DIR, f'checkpoint_{phase_name}.json')


def save_checkpoint(phase_name: str, results: list, derived_params: dict = None):
    """Faz sonuclarini Drive'a kaydet."""
    clean_results = []
    for r in results:
        r_clean = {}
        for k, v in r.items():
            if k == 'mask_snapshots':
                r_clean[k] = [
                    {'step': s['step'], 'mask': s['mask'].tolist()}
                    for s in v
                ]
            else:
                r_clean[k] = v
        clean_results.append(r_clean)

    data = {
        'phase': phase_name,
        'completed_at': datetime.now().isoformat(),
        'n_runs': len(results),
        'results': clean_results,
        'derived_params': derived_params or {},
    }
    path = checkpoint_path(phase_name)
    with open(path, 'w') as f:
        json.dump(data, f, indent=2, default=str)
    print(f'  [SAVED] {phase_name} -> {path} ({len(results)} runs)')


def load_checkpoint(phase_name: str) -> dict | None:
    """Drive'dan checkpoint yukle. Yoksa None dondur."""
    path = checkpoint_path(phase_name)
    if os.path.exists(path):
        with open(path) as f:
            data = json.load(f)
        print(f'  [RESUME] {phase_name} Drive\'dan yuklendi ({data["n_runs"]} runs, {data["completed_at"]})')
        return data
    return None


def save_partial(phase_name: str, results_so_far: list):
    """Faz icinde her run sonunda partial kayit."""
    path = os.path.join(DRIVE_SAVE_DIR, f'partial_{phase_name}.json')
    clean = []
    for r in results_so_far:
        r_clean = {k: v for k, v in r.items() if k != 'mask_snapshots'}
        clean.append(r_clean)
    with open(path, 'w') as f:
        json.dump({'phase': phase_name, 'partial': True, 'n_runs': len(clean),
                   'saved_at': datetime.now().isoformat(), 'results': clean},
                  f, indent=2, default=str)


# ════════════════════════════════════════════════════
#  PIPELINE CONFIG — V6 (composite gate + low pTM)
# ════════════════════════════════════════════════════

# -- PDB --
INITIAL_PDB = '1AKE'
TARGET_PDB  = '4AKE'
CHAIN_ID    = 'A'

# -- Pipeline --
N_STEPS              = 20
COMBINATION_STRATEGY = 'autostop'
Z_MIXING_ALPHA       = 0.7
N_ANM_MODES          = 20
N_COMBINATIONS       = 20
MAX_COMBO_SIZE       = 3
DF                   = 0.6
DF_MIN               = 0.3
DF_MAX               = 3.0
ANM_CUTOFF           = 15.0
CONTACT_R_CUT        = 10.0
CONTACT_TAU          = 1.5
Z_DIRECTION          = 'plus'

# -- OF3 --
USE_MSA_SERVER        = True
USE_TEMPLATES         = False
NUM_ROLLOUT_STEPS     = None
NUM_DIFFUSION_SAMPLES = 1

# -- Confidence V6: composite gate --
USE_COMPOSITE_GATE         = True
COMPOSITE_GATE_THRESHOLD   = 0.55
GATE_W_CR                  = 0.45
GATE_W_PLDDT               = 0.30
GATE_W_RG                  = 0.15
GATE_W_PTM                 = 0.10

# -- Hard cutoffs (safety net only) --
CONFIDENCE_PTM_CUTOFF     = 0.15   # V6: eskiden 0.30, simdi sadece safety net
CONFIDENCE_PLDDT_CUTOFF   = 65.0
CONFIDENCE_RANKING_CUTOFF = 0.45   # composite gate acikken kullanilmaz
CONFIDENCE_RG_MAX         = 2.5
CONFIDENCE_RG_MIN         = 0.3
CONFIDENCE_CLASH_REJECT   = True
CONFIDENCE_RMSD_INIT_MAX  = 10.0

# -- Stall & drift --
ALPHA_DECAY               = 0.8
MAX_STALL                 = 8
ENABLE_BEST_ROLLBACK      = True
BEST_ROLLBACK_TM_DROP     = 0.40
ENABLE_ADAPTIVE_STOP      = True
ADAPTIVE_STOP_WINDOW      = 3

# -- Fallback --
ENABLE_FALLBACK           = True
FALLBACK_EXTENDED_ENABLED = True
AUTOSTOP_FALLBACK_LEVELS  = (0, 1, 4, 9)

# -- Autostop --
AUTOSTOP_V_MAGNITUDE = 1.0
AUTOSTOP_N_STEPS     = 5000
AUTOSTOP_BACK_OFF    = 2

# -- Converter --
DRIVE_DIR = '/content/drive/MyDrive/ANM-openfold3/checkpoints/finetune_msa'
CHECKPOINT_SELECTION = 'best_bf_r'

# ════════════════════════════════════════════════════
#  GRID SEARCH PARAMETRELERI
# ════════════════════════════════════════════════════
GRID_CUTOFFS     = [0.05, 0.1, 0.15, 0.2, 0.3]
GRID_ALPHA_BASES = [0.0, 0.05]
GRID_ALPHA_MAXES = [0.5, 0.7, 1.0]
GRID_MAPPINGS    = ['linear', 'sigmoid', 'step']
GRID_W_PAIRS     = [(0.3, 0.7), (0.5, 0.5), (0.7, 0.3)]
GRID_DIAG_BANDS  = [0, 1, 2, 3]

print('V6 Config ready.')
print(f'  Drive save dir: {DRIVE_SAVE_DIR}')
print(f'  Composite gate: threshold={COMPOSITE_GATE_THRESHOLD}')
print(f'    weights: cR={GATE_W_CR} pLDDT={GATE_W_PLDDT} Rg={GATE_W_RG} pTM={GATE_W_PTM}')
print(f'  Hard cutoffs: pTM>={CONFIDENCE_PTM_CUTOFF} pLDDT>={CONFIDENCE_PLDDT_CUTOFF}')

## 3. Converter + PDB + OF3

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import json as _json
import numpy as np
import torch
import time
import copy
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

# src importable olsun
_repo_root = '/content/ANM-openfold3'
if _repo_root not in sys.path:
    sys.path.insert(0, _repo_root)

for mod_name in list(sys.modules.keys()):
    if mod_name.startswith('src'):
        del sys.modules[mod_name]

from src.converter import PairContactConverter
from src.mode_drive import ModeDrivePipeline, ModeDriveConfig, compute_rmsd, tm_score
from src.composite_confidence import (
    CompositeWeights, compute_composite, composite_score_from_step,
    WEIGHT_PRESETS, THRESHOLD_PRESETS,
    normalize_ptm, normalize_plddt, normalize_pae, normalize_rg, normalize_contact_recon,
)
from src.selective_mixing import compute_change_score, compute_alpha_mask, selective_blend_z

# -- Checkpoint --
history_path = os.path.join(DRIVE_DIR, 'history.json')
best_model_path = os.path.join(DRIVE_DIR, 'best_model.pt')

if CHECKPOINT_SELECTION == 'best_bf_r' and os.path.exists(history_path):
    with open(history_path) as f:
        history = _json.load(f)
    best_bf_r = -1
    best_epoch = -1
    for entry in history:
        bf_r = entry.get('val_bf_pearson', 0)
        if bf_r > best_bf_r:
            best_bf_r = bf_r
            best_epoch = entry['epoch']
    epoch_ckpt = os.path.join(DRIVE_DIR, f'epoch_{best_epoch:04d}.pt')
    CHECKPOINT_PATH = epoch_ckpt if os.path.exists(epoch_ckpt) else best_model_path
    print(f'Best bf_r: {best_bf_r:.4f} at epoch {best_epoch}')
else:
    CHECKPOINT_PATH = best_model_path

converter = PairContactConverter(CHECKPOINT_PATH)
print(f'Converter loaded from {CHECKPOINT_PATH}')

In [ ]:
# -- PDB --
from Bio.PDB import PDBParser, PDBList
from Bio.SeqUtils import seq1

def fetch_ca(pdb_id, chain_id):
    parser = PDBParser(QUIET=True)
    pdbl = PDBList()
    pdb_file = pdbl.retrieve_pdb_file(pdb_id, pdir='/tmp/pdb_cache', file_format='pdb')
    structure = parser.get_structure(pdb_id, pdb_file)
    chain = [c for c in structure[0].get_chains() if c.id == chain_id][0]
    residues = [r for r in chain if r.get_id()[0] == ' ' and 'CA' in r]
    coords = torch.tensor([r['CA'].get_vector().get_array() for r in residues], dtype=torch.float32)
    sequence = ''.join(seq1(r.get_resname()) for r in residues)
    return coords, sequence

initial_ca, sequence = fetch_ca(INITIAL_PDB, CHAIN_ID)
target_ca, _ = fetch_ca(TARGET_PDB, CHAIN_ID)
N = initial_ca.shape[0]
print(f'Initial: {INITIAL_PDB} chain {CHAIN_ID}, N={N}')
print(f'Target:  {TARGET_PDB} chain {CHAIN_ID}, N={target_ca.shape[0]}')
print(f'RMSD to target: {compute_rmsd(initial_ca, target_ca):.2f} A')
print(f'TM-score: {tm_score(initial_ca, target_ca):.4f}')

# -- OF3 --
_query_dir = '/content/of3_queries'
os.makedirs(_query_dir, exist_ok=True)
_query = {
    'queries': {
        INITIAL_PDB: {
            'chains': [{
                'molecule_type': 'protein',
                'chain_ids': [CHAIN_ID],
                'sequence': sequence,
            }]
        }
    }
}
_query_path = os.path.join(_query_dir, f'{INITIAL_PDB}.json')
with open(_query_path, 'w') as f:
    _json.dump(_query, f, indent=2)

from src.of3_diffusion import load_of3_diffusion
diffusion_fn, zij_trunk = load_of3_diffusion(
    query_json=_query_path,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    use_msa_server=USE_MSA_SERVER,
    use_templates=USE_TEMPLATES,
    num_rollout_steps=NUM_ROLLOUT_STEPS,
    num_samples=NUM_DIFFUSION_SAMPLES,
)
print(f'zij_trunk shape: {zij_trunk.shape}')

from src.autostop_adapter import StructureContext
_pdb_file = PDBList().retrieve_pdb_file(INITIAL_PDB, pdir='/tmp/pdb_cache', file_format='pdb')
structure_ctx = StructureContext.from_pdb(_pdb_file, chain_id=CHAIN_ID)
print(f'StructureContext: N={structure_ctx.struct.N}')
print('OF3 ready.')

## 4. V6 Runner

In [ ]:
# ════════════════════════════════════════════════════
#  V6 RUNNER — composite gate + selective mixing + diagonal band
# ════════════════════════════════════════════════════

def make_v6_config(
    n_steps=N_STEPS,
    alpha=Z_MIXING_ALPHA,
    alpha_decay=ALPHA_DECAY,
    max_stall=MAX_STALL,
    selective_mixing=False,
    selective_params=None,
    diagonal_band=1,
):
    sel = selective_params or {}
    return ModeDriveConfig(
        n_steps=n_steps,
        combination_strategy=COMBINATION_STRATEGY,
        z_mixing_alpha=alpha,
        n_anm_modes=N_ANM_MODES,
        n_combinations=N_COMBINATIONS,
        max_combo_size=MAX_COMBO_SIZE,
        df=DF, df_min=DF_MIN, df_max=DF_MAX,
        anm_cutoff=ANM_CUTOFF,
        contact_r_cut=CONTACT_R_CUT,
        contact_tau=CONTACT_TAU,
        z_direction=Z_DIRECTION,
        num_diffusion_samples=NUM_DIFFUSION_SAMPLES,
        # Composite gate
        use_composite_gate=USE_COMPOSITE_GATE,
        composite_gate_threshold=COMPOSITE_GATE_THRESHOLD,
        gate_w_cr=GATE_W_CR,
        gate_w_plddt=GATE_W_PLDDT,
        gate_w_rg=GATE_W_RG,
        gate_w_ptm=GATE_W_PTM,
        # Hard cutoffs
        confidence_ptm_cutoff=CONFIDENCE_PTM_CUTOFF,
        confidence_plddt_cutoff=CONFIDENCE_PLDDT_CUTOFF,
        confidence_ranking_cutoff=CONFIDENCE_RANKING_CUTOFF,
        confidence_rg_max=CONFIDENCE_RG_MAX,
        confidence_rg_min=CONFIDENCE_RG_MIN,
        confidence_clash_reject=CONFIDENCE_CLASH_REJECT,
        confidence_rmsd_init_max=CONFIDENCE_RMSD_INIT_MAX,
        # Fallback & drift
        enable_confidence_fallback=ENABLE_FALLBACK,
        fallback_extended_enabled=FALLBACK_EXTENDED_ENABLED,
        autostop_v_magnitude=AUTOSTOP_V_MAGNITUDE,
        autostop_n_steps=AUTOSTOP_N_STEPS,
        autostop_back_off=AUTOSTOP_BACK_OFF,
        autostop_fallback_levels=AUTOSTOP_FALLBACK_LEVELS,
        autostop_chain_id=CHAIN_ID,
        rejected_alpha_decay=alpha_decay,
        max_consecutive_rejected=max_stall,
        confidence_warmup_steps=0,
        enable_best_rollback=ENABLE_BEST_ROLLBACK,
        best_rollback_tm_drop=BEST_ROLLBACK_TM_DROP,
        enable_adaptive_stop=ENABLE_ADAPTIVE_STOP,
        adaptive_stop_window=ADAPTIVE_STOP_WINDOW,
        # V6 selective mixing + diagonal band
        selective_mixing=selective_mixing,
        selective_w_connectivity=sel.get('w_connectivity', 0.5),
        selective_w_distance=sel.get('w_distance', 0.5),
        selective_change_cutoff=sel.get('change_cutoff', 0.1),
        selective_alpha_base=sel.get('alpha_base', 0.0),
        selective_alpha_max=sel.get('alpha_max', 1.0),
        selective_mapping=sel.get('mapping', 'linear'),
        selective_distance_mode=sel.get('distance_mode', 'max'),
        selective_diagonal_band=diagonal_band,
    )


def run_v6(
    label: str = '',
    n_steps: int = N_STEPS,
    alpha: float = Z_MIXING_ALPHA,
    alpha_decay: float = ALPHA_DECAY,
    max_stall: int = MAX_STALL,
    early_abort_steps: int = 5,
    selective_mixing: bool = False,
    selective_params: dict | None = None,
    diagonal_band: int = 1,
    verbose: bool = True,
) -> dict:
    """V6 runner: composite gate, diagonal band, selective mixing."""
    cfg = make_v6_config(
        n_steps=n_steps, alpha=alpha,
        alpha_decay=alpha_decay, max_stall=max_stall,
        selective_mixing=selective_mixing,
        selective_params=selective_params,
        diagonal_band=diagonal_band,
    )
    pipe = ModeDrivePipeline(
        converter=converter, config=cfg,
        diffusion_fn=diffusion_fn,
        structure_ctx=structure_ctx,
    )

    sel_tag = 'SELECTIVE' if selective_mixing else 'UNIFORM'
    sel_info = ''
    if selective_mixing and selective_params:
        sp = selective_params
        sel_info = (
            f'  selective: cutoff={sp.get("change_cutoff", 0.1)} '
            f'base={sp.get("alpha_base", 0.0)} '
            f'max={sp.get("alpha_max", 1.0)} '
            f'map={sp.get("mapping", "linear")} '
            f'w=({sp.get("w_connectivity", 0.5)},{sp.get("w_distance", 0.5)}) '
            f'band={diagonal_band}'
        )

    print(f'\n{"="*70}')
    print(f'  {label} [{sel_tag}]')
    if sel_info:
        print(sel_info)
    print(f'{"="*70}')

    coords_ca = initial_ca.clone()
    z_current = zij_trunk.clone()
    orig_alpha = alpha
    current_alpha = alpha
    consecutive_rejected = 0

    coords_best = initial_ca.clone()
    z_best = zij_trunk.clone()
    tm_best = 0.0
    best_step_idx = -1
    rollback_count = 0
    accepted_tm_history = []

    step_metrics = []
    mask_snapshots = []
    t0 = time.time()
    stopped_early = False

    for step_idx in range(n_steps):
        cfg.z_mixing_alpha = current_alpha
        sr = pipe.step_with_fallback(
            coords_ca, initial_ca, z_current,
            prev_rmsd=0.0,
            target_coords=target_ca,
            step_idx=step_idx,
        )

        if sr.alpha_mask_snapshot is not None:
            mask_snapshots.append({
                'step': step_idx + 1,
                'mask': sr.alpha_mask_snapshot.numpy(),
            })

        # V6: confidence is handled inside pipeline via composite gate
        # Here we just check the result's rejected flag
        accepted = not sr.rejected
        reject_reason = ''

        # Additional hard rejects (safety net)
        if sr.rg_ratio is not None and sr.rg_ratio > CONFIDENCE_RG_MAX:
            accepted = False
            reject_reason = f'Rg={sr.rg_ratio:.1f}>{CONFIDENCE_RG_MAX}'
        if sr.has_clash and CONFIDENCE_CLASH_REJECT:
            accepted = False
            reject_reason = 'clash'
        if sr.rmsd > CONFIDENCE_RMSD_INIT_MAX:
            accepted = False
            reject_reason = f'rmsd_init={sr.rmsd:.1f}>{CONFIDENCE_RMSD_INIT_MAX}'

        rmsd_tgt = compute_rmsd(sr.new_ca, target_ca)
        tm_tgt = tm_score(sr.new_ca, target_ca)

        m = {
            'step': step_idx + 1,
            'accepted': accepted,
            'rmsd_init': sr.rmsd,
            'rmsd_tgt': rmsd_tgt,
            'tm_tgt': tm_tgt,
            'ptm': sr.ptm,
            'plddt_mean': float(sr.plddt.mean()) if sr.plddt is not None else None,
            'ranking': sr.ranking_score,
            'mean_pae': sr.mean_pae,
            'rg_ratio': sr.rg_ratio,
            'contact_recon': sr.contact_recon,
            'has_clash': sr.has_clash,
            'fallback_level': sr.fallback_level,
            'alpha_used': current_alpha,
            'reject_reason': reject_reason,
            'rollback': False,
            'change_score_mean': sr.change_score_mean,
            'change_score_max': sr.change_score_max,
            'n_active_pairs': sr.n_active_pairs,
            'alpha_mask_mean': sr.alpha_mask_mean,
        }

        if verbose:
            tag = 'ACC' if accepted else 'REJ'
            plddt_str = f"{float(sr.plddt.mean()):.0f}" if sr.plddt is not None else '?'
            cr_str = f"{sr.contact_recon:.2f}" if sr.contact_recon is not None else '?'
            sel_diag = ''
            if selective_mixing and sr.change_score_mean is not None:
                sel_diag = f' cs={sr.change_score_mean:.3f} np={sr.n_active_pairs}'
            extra = f' {reject_reason}' if not accepted else ''
            print(
                f'  [{step_idx+1:>2}/{n_steps}] {tag}  '
                f'TM={tm_tgt:.3f} RMSD={rmsd_tgt:.2f} '
                f'pTM={sr.ptm:.3f} pLDDT={plddt_str} cR={cr_str} '
                f'a={current_alpha:.3f} FL={sr.fallback_level}'
                f'{sel_diag}{extra}'
            )

        if accepted:
            coords_ca = sr.new_ca
            z_current = sr.z_modified
            consecutive_rejected = 0
            current_alpha = orig_alpha

            if tm_tgt > tm_best:
                tm_best = tm_tgt
                coords_best = sr.new_ca.clone()
                z_best = sr.z_modified.clone()
                best_step_idx = step_idx
            elif tm_best > 0 and tm_tgt < tm_best * (1.0 - BEST_ROLLBACK_TM_DROP):
                if verbose:
                    print(f'  >> ROLLBACK to step {best_step_idx+1}')
                coords_ca = coords_best.clone()
                z_current = z_best.clone()
                rollback_count += 1
                m['rollback'] = True

            accepted_tm_history.append(tm_tgt)
            if ENABLE_ADAPTIVE_STOP and len(accepted_tm_history) >= ADAPTIVE_STOP_WINDOW + 1:
                recent = accepted_tm_history[-(ADAPTIVE_STOP_WINDOW + 1):]
                if all(recent[i] > recent[i+1] for i in range(len(recent)-1)):
                    if verbose:
                        print(f'  >> EARLY STOP: {ADAPTIVE_STOP_WINDOW} ardisik TM dusus')
                    stopped_early = True
                    step_metrics.append(m)
                    break
        else:
            consecutive_rejected += 1
            if alpha_decay < 1.0:
                current_alpha = max(0.02, current_alpha * alpha_decay)
            if max_stall > 0 and consecutive_rejected >= max_stall:
                if verbose:
                    print(f'  STALL: {consecutive_rejected} consecutive rejected')
                step_metrics.append(m)
                break

        step_metrics.append(m)

        if step_idx + 1 == early_abort_steps:
            n_acc = sum(1 for x in step_metrics if x['accepted'])
            if n_acc == 0:
                if verbose:
                    print(f'  EARLY ABORT: 0/{early_abort_steps} accepted')
                break

    wall = time.time() - t0
    total_steps = len(step_metrics)
    n_accepted = sum(1 for x in step_metrics if x['accepted'])

    accepted_metrics = [x for x in step_metrics if x['accepted']]
    if accepted_metrics:
        best_step = max(accepted_metrics, key=lambda x: x['tm_tgt'])
        best_tm = best_step['tm_tgt']
        best_rmsd = best_step['rmsd_tgt']
    else:
        best_tm = tm_score(initial_ca, target_ca)
        best_rmsd = compute_rmsd(initial_ca, target_ca)

    result = {
        'label': label,
        'alpha': alpha,
        'alpha_decay': alpha_decay,
        'selective_mixing': selective_mixing,
        'selective_params': selective_params or {},
        'diagonal_band': diagonal_band,
        'total_steps': total_steps,
        'accepted': n_accepted,
        'rejected': total_steps - n_accepted,
        'accept_rate': n_accepted / max(1, total_steps),
        'best_tm': best_tm,
        'best_rmsd': best_rmsd,
        'rollback_count': rollback_count,
        'stopped_early': stopped_early,
        'wall_s': wall,
        'step_metrics': step_metrics,
        'mask_snapshots': mask_snapshots,
    }

    print(f'  => {n_accepted}/{total_steps} acc  '
          f'best_TM={best_tm:.4f}  best_RMSD={best_rmsd:.2f}  wall={wall:.0f}s')

    # Inline heatmaps (selective only)
    if selective_mixing and mask_snapshots and verbose:
        n_show = min(len(mask_snapshots), 4)
        indices = np.linspace(0, len(mask_snapshots)-1, n_show, dtype=int)
        fig, axes = plt.subplots(1, n_show, figsize=(3.5*n_show, 3))
        if n_show == 1:
            axes = [axes]
        for ax, idx in zip(axes, indices):
            snap = mask_snapshots[idx]
            ax.imshow(snap['mask'], cmap='hot', vmin=0, vmax=1, aspect='equal')
            ax.set_title(f'Step {snap["step"]}', fontsize=9)
        fig.suptitle(f'{label}', fontsize=10)
        plt.tight_layout()
        plt.show()

    return result


print('V6 Runner ready.')

## 5. Phase A: Uniform vs Selective Baseline (2 run)

Her ikisini de default parametrelerle calistir — composite gate'in etkisini gor.

In [ ]:
# ════════════════════════════════════════════════════
#  PHASE A: UNIFORM vs SELECTIVE BASELINE
# ════════════════════════════════════════════════════

ckpt_a = load_checkpoint('phase_a')

if ckpt_a is not None:
    RESULTS_A = ckpt_a['results']
else:
    RESULTS_A = []

    # A1: Uniform (no selective mixing)
    r_uniform = run_v6(
        label='A_uniform',
        selective_mixing=False,
    )
    RESULTS_A.append(r_uniform)
    save_partial('phase_a', RESULTS_A)

    # A2: Selective (default params)
    r_selective = run_v6(
        label='A_selective_default',
        selective_mixing=True,
        selective_params={
            'change_cutoff': 0.1,
            'alpha_base': 0.0,
            'alpha_max': 1.0,
            'mapping': 'linear',
            'w_connectivity': 0.5,
            'w_distance': 0.5,
        },
    )
    RESULTS_A.append(r_selective)

    save_checkpoint('phase_a', RESULTS_A)

# Sonuc
for r in RESULTS_A:
    tag = 'SEL' if r.get('selective_mixing') else 'UNI'
    print(f"  {r['label']:30s} [{tag}]  TM={r['best_tm']:.4f}  RMSD={r['best_rmsd']:.2f}  acc={r['accepted']}/{r['total_steps']}")

delta_tm = RESULTS_A[1]['best_tm'] - RESULTS_A[0]['best_tm'] if len(RESULTS_A) == 2 else 0
print(f'\nDelta TM (selective - uniform): {delta_tm:+.4f}')

## 6. Phase B: change_cutoff Sweep (5 run)

Cutoff: [0.05, 0.1, 0.15, 0.2, 0.3] — diger parametreler default.

In [ ]:
# ════════════════════════════════════════════════════
#  PHASE B: CHANGE_CUTOFF SWEEP
# ════════════════════════════════════════════════════

ckpt_b = load_checkpoint('phase_b')

if ckpt_b is not None:
    RESULTS_B = ckpt_b['results']
    BEST_CUTOFF = ckpt_b['derived_params']['BEST_CUTOFF']
else:
    RESULTS_B = []

    for cutoff in GRID_CUTOFFS:
        label = f'B_cutoff_{cutoff:.2f}'
        r = run_v6(
            label=label,
            selective_mixing=True,
            selective_params={
                'change_cutoff': cutoff,
                'alpha_base': 0.0,
                'alpha_max': 1.0,
                'mapping': 'linear',
                'w_connectivity': 0.5,
                'w_distance': 0.5,
            },
        )
        RESULTS_B.append(r)
        save_partial('phase_b', RESULTS_B)

    best_b = max(RESULTS_B, key=lambda r: (r['best_tm'], r['accepted']))
    BEST_CUTOFF = best_b['selective_params']['change_cutoff']
    save_checkpoint('phase_b', RESULTS_B, {'BEST_CUTOFF': BEST_CUTOFF})

# Sonuc tablosu
rows = []
for r in RESULTS_B:
    sp = r.get('selective_params', {})
    rows.append({
        'cutoff': sp.get('change_cutoff'),
        'accepted': r['accepted'],
        'total': r['total_steps'],
        'best_TM': f"{r['best_tm']:.4f}",
        'best_RMSD': f"{r['best_rmsd']:.2f}",
        'early_stop': r.get('stopped_early', False),
    })
print('\nPHASE B SONUC:')
print(pd.DataFrame(rows).to_string(index=False))
print(f'\n=> BEST_CUTOFF = {BEST_CUTOFF}')

## 7. Phase C: alpha_base x alpha_max (6 run)

Best cutoff ile: base=[0.0, 0.05] x max=[0.5, 0.7, 1.0]

In [ ]:
# ════════════════════════════════════════════════════
#  PHASE C: ALPHA_BASE x ALPHA_MAX
# ════════════════════════════════════════════════════

ckpt_c = load_checkpoint('phase_c')

if ckpt_c is not None:
    RESULTS_C = ckpt_c['results']
    BEST_ALPHA_BASE = ckpt_c['derived_params']['BEST_ALPHA_BASE']
    BEST_ALPHA_MAX = ckpt_c['derived_params']['BEST_ALPHA_MAX']
else:
    RESULTS_C = []

    for base in GRID_ALPHA_BASES:
        for amax in GRID_ALPHA_MAXES:
            label = f'C_base{base:.2f}_max{amax:.1f}'
            r = run_v6(
                label=label,
                selective_mixing=True,
                selective_params={
                    'change_cutoff': BEST_CUTOFF,
                    'alpha_base': base,
                    'alpha_max': amax,
                    'mapping': 'linear',
                    'w_connectivity': 0.5,
                    'w_distance': 0.5,
                },
            )
            RESULTS_C.append(r)
            save_partial('phase_c', RESULTS_C)

    best_c = max(RESULTS_C, key=lambda r: (r['best_tm'], r['accepted']))
    BEST_ALPHA_BASE = best_c['selective_params']['alpha_base']
    BEST_ALPHA_MAX = best_c['selective_params']['alpha_max']
    save_checkpoint('phase_c', RESULTS_C, {
        'BEST_ALPHA_BASE': BEST_ALPHA_BASE,
        'BEST_ALPHA_MAX': BEST_ALPHA_MAX,
    })

# Sonuc tablosu
rows = []
for r in RESULTS_C:
    sp = r.get('selective_params', {})
    rows.append({
        'base': sp.get('alpha_base'),
        'max': sp.get('alpha_max'),
        'accepted': r['accepted'],
        'total': r['total_steps'],
        'best_TM': f"{r['best_tm']:.4f}",
        'best_RMSD': f"{r['best_rmsd']:.2f}",
    })
print('\nPHASE C SONUC:')
print(pd.DataFrame(rows).to_string(index=False))
print(f'\n=> BEST_ALPHA_BASE={BEST_ALPHA_BASE}, BEST_ALPHA_MAX={BEST_ALPHA_MAX}')

## 8. Phase D: Mapping & Weight Sweep (9 run)

Best cutoff + alpha ile: mapping=[linear, sigmoid, step] x w=[(0.3,0.7), (0.5,0.5), (0.7,0.3)]

In [ ]:
# ════════════════════════════════════════════════════
#  PHASE D: MAPPING & WEIGHT SWEEP
# ════════════════════════════════════════════════════

ckpt_d = load_checkpoint('phase_d')

if ckpt_d is not None:
    RESULTS_D = ckpt_d['results']
    BEST_MAPPING = ckpt_d['derived_params']['BEST_MAPPING']
    BEST_W_C = ckpt_d['derived_params']['BEST_W_C']
    BEST_W_D = ckpt_d['derived_params']['BEST_W_D']
else:
    RESULTS_D = []

    for mapping in GRID_MAPPINGS:
        for w_c, w_d in GRID_W_PAIRS:
            label = f'D_{mapping}_wc{w_c:.1f}_wd{w_d:.1f}'
            r = run_v6(
                label=label,
                selective_mixing=True,
                selective_params={
                    'change_cutoff': BEST_CUTOFF,
                    'alpha_base': BEST_ALPHA_BASE,
                    'alpha_max': BEST_ALPHA_MAX,
                    'mapping': mapping,
                    'w_connectivity': w_c,
                    'w_distance': w_d,
                },
            )
            RESULTS_D.append(r)
            save_partial('phase_d', RESULTS_D)

    best_d = max(RESULTS_D, key=lambda r: (r['best_tm'], r['accepted']))
    BEST_MAPPING = best_d['selective_params']['mapping']
    BEST_W_C = best_d['selective_params']['w_connectivity']
    BEST_W_D = best_d['selective_params']['w_distance']
    save_checkpoint('phase_d', RESULTS_D, {
        'BEST_MAPPING': BEST_MAPPING,
        'BEST_W_C': BEST_W_C,
        'BEST_W_D': BEST_W_D,
    })

# Sonuc tablosu
rows = []
for r in RESULTS_D:
    sp = r.get('selective_params', {})
    rows.append({
        'mapping': sp.get('mapping'),
        'w_c': sp.get('w_connectivity'),
        'w_d': sp.get('w_distance'),
        'accepted': r['accepted'],
        'total': r['total_steps'],
        'best_TM': f"{r['best_tm']:.4f}",
        'best_RMSD': f"{r['best_rmsd']:.2f}",
    })
print('\nPHASE D SONUC:')
print(pd.DataFrame(rows).to_string(index=False))
print(f'\n=> BEST_MAPPING={BEST_MAPPING}, BEST_W_C={BEST_W_C}, BEST_W_D={BEST_W_D}')

## 9. Phase E: Diagonal Band Sweep (4 run)

Best parametreler ile: diagonal_band = [0, 1, 2, 3]

- band=0: sadece diagonal (i==j) korunur
- band=1: diagonal + +-1 (default)
- band=2: +-2 backbone neighbors
- band=3: +-3 local geometry

In [ ]:
# ════════════════════════════════════════════════════
#  PHASE E: DIAGONAL BAND SWEEP
# ════════════════════════════════════════════════════

ckpt_e = load_checkpoint('phase_e')

if ckpt_e is not None:
    RESULTS_E = ckpt_e['results']
    BEST_DIAG_BAND = ckpt_e['derived_params']['BEST_DIAG_BAND']
else:
    RESULTS_E = []

    for band in GRID_DIAG_BANDS:
        label = f'E_band_{band}'
        r = run_v6(
            label=label,
            selective_mixing=True,
            selective_params={
                'change_cutoff': BEST_CUTOFF,
                'alpha_base': BEST_ALPHA_BASE,
                'alpha_max': BEST_ALPHA_MAX,
                'mapping': BEST_MAPPING,
                'w_connectivity': BEST_W_C,
                'w_distance': BEST_W_D,
            },
            diagonal_band=band,
        )
        RESULTS_E.append(r)
        save_partial('phase_e', RESULTS_E)

    best_e = max(RESULTS_E, key=lambda r: (r['best_tm'], r['accepted']))
    BEST_DIAG_BAND = best_e['diagonal_band']
    save_checkpoint('phase_e', RESULTS_E, {'BEST_DIAG_BAND': BEST_DIAG_BAND})

# Sonuc tablosu
rows = []
for r in RESULTS_E:
    rows.append({
        'band': r['diagonal_band'],
        'accepted': r['accepted'],
        'total': r['total_steps'],
        'best_TM': f"{r['best_tm']:.4f}",
        'best_RMSD': f"{r['best_rmsd']:.2f}",
    })
print('\nPHASE E SONUC:')
print(pd.DataFrame(rows).to_string(index=False))
print(f'\n=> BEST_DIAG_BAND = {BEST_DIAG_BAND}')

## 10. Final Analysis

In [ ]:
# ════════════════════════════════════════════════════
#  ANALYSIS — GENEL SONUC
# ════════════════════════════════════════════════════

ALL_RESULTS = {
    'phase_a': RESULTS_A,
    'phase_b': RESULTS_B,
    'phase_c': RESULTS_C,
    'phase_d': RESULTS_D,
    'phase_e': RESULTS_E,
}

# Genel tablo
all_rows = []
for phase_name, results in ALL_RESULTS.items():
    for r in results:
        sp = r.get('selective_params', {})
        all_rows.append({
            'phase': phase_name,
            'label': r['label'],
            'selective': r.get('selective_mixing', False),
            'best_TM': r['best_tm'],
            'best_RMSD': r['best_rmsd'],
            'accepted': r['accepted'],
            'total': r['total_steps'],
            'band': r.get('diagonal_band', 1),
        })
df_all = pd.DataFrame(all_rows).sort_values('best_TM', ascending=False)
print('='*90)
print('GENEL SONUC TABLOSU (sorted by TM)')
print('='*90)
print(df_all.to_string(index=False))

# En iyi genel
all_flat = []
for results in ALL_RESULTS.values():
    all_flat.extend(results)
best_overall = max(all_flat, key=lambda r: (r['best_tm'], r['accepted']))
print(f'\nEN IYI: {best_overall["label"]}')
print(f'  TM={best_overall["best_tm"]:.4f}  RMSD={best_overall["best_rmsd"]:.2f}')
print(f'  accepted={best_overall["accepted"]}/{best_overall["total_steps"]}')

if best_overall.get('selective_mixing'):
    sp = best_overall.get('selective_params', {})
    print(f'\nEn iyi selective params:')
    print(f'  change_cutoff    = {sp.get("change_cutoff", "-")}')
    print(f'  alpha_base       = {sp.get("alpha_base", "-")}')
    print(f'  alpha_max        = {sp.get("alpha_max", "-")}')
    print(f'  mapping          = {sp.get("mapping", "-")}')
    print(f'  w_connectivity   = {sp.get("w_connectivity", "-")}')
    print(f'  w_distance       = {sp.get("w_distance", "-")}')
    print(f'  diagonal_band    = {best_overall.get("diagonal_band", 1)}')

print(f'\nFinal best params for ModeDriveConfig:')
print(f'  BEST_CUTOFF      = {BEST_CUTOFF}')
print(f'  BEST_ALPHA_BASE  = {BEST_ALPHA_BASE}')
print(f'  BEST_ALPHA_MAX   = {BEST_ALPHA_MAX}')
print(f'  BEST_MAPPING     = {BEST_MAPPING}')
print(f'  BEST_W_C         = {BEST_W_C}')
print(f'  BEST_W_D         = {BEST_W_D}')
print(f'  BEST_DIAG_BAND   = {BEST_DIAG_BAND}')

In [ ]:
# ── Grafikler ──

# 1. TM-score bar chart (tum runlar)
fig, ax = plt.subplots(figsize=(12, max(6, len(all_flat)*0.28)))
labels_plot = [r['label'] for r in sorted(all_flat, key=lambda x: x['best_tm'])]
tms = [r['best_tm'] for r in sorted(all_flat, key=lambda x: x['best_tm'])]
colors = ['#e74c3c' if not r.get('selective_mixing') else '#3498db'
           for r in sorted(all_flat, key=lambda x: x['best_tm'])]
ax.barh(labels_plot, tms, color=colors)
ax.axvline(x=RESULTS_A[0]['best_tm'] if RESULTS_A else 0, color='red', linestyle='--', alpha=0.5, label='Uniform baseline')
ax.set_xlabel('Best TM-score')
ax.set_title('V6 Grid Search — All Runs (red=uniform, blue=selective)')
ax.legend()
plt.tight_layout()
plt.show()

# 2. Phase progression
phase_bests = {}
for phase_name, results in ALL_RESULTS.items():
    if results:
        best_r = max(results, key=lambda r: r['best_tm'])
        phase_bests[phase_name] = best_r['best_tm']

fig, ax = plt.subplots(figsize=(8, 4))
phases = list(phase_bests.keys())
bests = list(phase_bests.values())
ax.plot(phases, bests, 'bo-', markersize=10)
ax.set_ylabel('Best TM-score')
ax.set_title('Best TM per Phase')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 3. Best run step trajectory
if best_overall.get('step_metrics'):
    metrics = best_overall['step_metrics']
    steps = [m['step'] for m in metrics]
    tm_vals = [m['tm_tgt'] for m in metrics]
    accepted_flags = [m['accepted'] for m in metrics]

    fig, ax = plt.subplots(figsize=(12, 4))
    for s, t, a in zip(steps, tm_vals, accepted_flags):
        color = '#2ecc71' if a else '#e74c3c'
        ax.scatter(s, t, color=color, s=80, zorder=5, edgecolors='black', linewidths=0.5)
    ax.plot(steps, tm_vals, 'k-', alpha=0.3)
    ax.set_xlabel('Step')
    ax.set_ylabel('TM-score vs target')
    ax.set_title(f'Best run trajectory: {best_overall["label"]}')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
# ════════════════════════════════════════════════════
#  FINAL SAVE TO DRIVE
# ════════════════════════════════════════════════════

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

summary_rows = []
for phase_name, results in ALL_RESULTS.items():
    for r in results:
        sp = r.get('selective_params', {})
        summary_rows.append({
            'phase': phase_name,
            'label': r['label'],
            'selective_mixing': r.get('selective_mixing', False),
            'change_cutoff': sp.get('change_cutoff', ''),
            'alpha_base': sp.get('alpha_base', ''),
            'alpha_max': sp.get('alpha_max', ''),
            'mapping': sp.get('mapping', ''),
            'w_connectivity': sp.get('w_connectivity', ''),
            'w_distance': sp.get('w_distance', ''),
            'diagonal_band': r.get('diagonal_band', 1),
            'best_tm': r['best_tm'],
            'best_rmsd': r['best_rmsd'],
            'accepted': r['accepted'],
            'total_steps': r['total_steps'],
            'accept_rate': r.get('accept_rate', 0),
            'rollbacks': r.get('rollback_count', 0),
            'early_stop': r.get('stopped_early', False),
            'wall_s': r.get('wall_s', 0),
        })

summary_path = os.path.join(DRIVE_SAVE_DIR, f'summary_v6_{timestamp}.json')
with open(summary_path, 'w') as f:
    json.dump({
        'completed_at': datetime.now().isoformat(),
        'total_runs': len(summary_rows),
        'best_label': best_overall['label'],
        'best_tm': best_overall['best_tm'],
        'best_rmsd': best_overall['best_rmsd'],
        'best_params': {
            **best_overall.get('selective_params', {}),
            'diagonal_band': best_overall.get('diagonal_band', 1),
        },
        'derived': {
            'BEST_CUTOFF': BEST_CUTOFF,
            'BEST_ALPHA_BASE': BEST_ALPHA_BASE,
            'BEST_ALPHA_MAX': BEST_ALPHA_MAX,
            'BEST_MAPPING': BEST_MAPPING,
            'BEST_W_C': BEST_W_C,
            'BEST_W_D': BEST_W_D,
            'BEST_DIAG_BAND': BEST_DIAG_BAND,
        },
        'results': summary_rows,
    }, f, indent=2, default=str)
print(f'Summary JSON: {summary_path}')

csv_path = os.path.join(DRIVE_SAVE_DIR, f'summary_v6_{timestamp}.csv')
pd.DataFrame(summary_rows).to_csv(csv_path, index=False)
print(f'Summary CSV:  {csv_path}')

print(f'\nTamamlandi! {len(summary_rows)} run sonucu Drive\'a kaydedildi.')
print(f'Drive dir: {DRIVE_SAVE_DIR}')